# 5.4. Stage 5: Monthly Aggregation — Full Dataset

**Stage:** 5.4 of 5

**Purpose:** Stream completed daily full Parquet files from Stage 5.3 into monthly full Parquet datasets without loading an entire month into memory.

**Output:**
- Monthly full Parquet files
- Monthly Ukraine-country JSON files (`country == "Ukraine"`)

Months are derived automatically from completed tracker rows. Monthly Parquet is first written to a temporary file and moved into place only after a successful close.

**Documentation:** [Notebook guide](README.md) · [Stage data description](../../data/data-pipeline/stage_05/README_data.md)


## 5.4.1. Load configuration and dependencies

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys

import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

from pipeline_bootstrap import configure_pipeline
configure_pipeline()
import general as g

g.clean_memory()

## 5.4.2. Load tracker and discover months

In [ ]:
process_df = pd.read_pickle(g.Config.STAGE5_PROCESS_FULL_PATH)
process_df = process_df.sort_values("input_path").reset_index(drop=True)
process_df["input_date"] = pd.to_datetime(
    process_df["input_path"].astype(str).str.extract(r"(\d{4}-\d{2}-\d{2})", expand=False),
    errors="coerce",
)

invalid_date_count = process_df["input_date"].isna().sum()
if invalid_date_count:
    print(f"Skipping {invalid_date_count} tracker rows with invalid input dates.")

completed_process_df = process_df[
    process_df["rejoin_status"].eq("complete")
    & process_df["input_date"].notna()
].copy()

months_df = (
    completed_process_df["input_date"]
    .dt.to_period("M")
    .drop_duplicates()
    .sort_values()
    .to_frame(name="period")
    .reset_index(drop=True)
)
months_df["year"] = months_df["period"].dt.year
months_df["month"] = months_df["period"].dt.month

g.check_folder_exists(g.Config.STAGE5_MONTHLY_FULL_PARQUET_PATH)
g.check_folder_exists(g.Config.STAGE5_MONTHLY_FULL_JSON_UA_PATH)

print(f"Completed daily full files: {completed_process_df.shape[0]}")
print(f"Calendar months to aggregate: {months_df.shape[0]}")

## 5.4.3. Define monthly schema helpers

In [ ]:
selected_columns = [
    "id", "title", "description", "min_salary", "max_salary", "currency",
    "salary_rate", "date_created", "date_expired", "clean_title", "clean_desc",
    "title_lang", "desc_lang", "skill_ids", "skill_labels", "job_type",
    "classified_code", "classified_title", "skill_labels_en",
    "classified_title_clean", "extract_type", "esco_title", "esco_skills",
    "esco_id", "esco_code", "number_of_clicks", "date", "region_original",
    "city", "district", "region", "country", "latitude", "longitude",
]


def selected_schema(path):
    schema = pq.read_schema(path)
    fields = [
        schema.field(column) if column in schema.names else pa.field(column, pa.null())
        for column in selected_columns
    ]
    return pa.schema(fields)


def unify_selected_schemas(schemas):
    try:
        return pa.unify_schemas(schemas, promote_options="permissive")
    except TypeError:
        return pa.unify_schemas(schemas)


def align_table_to_schema(table, target_schema):
    arrays = []
    for field in target_schema:
        if field.name not in table.schema.names:
            arrays.append(pa.nulls(table.num_rows, type=field.type))
            continue

        column = table[field.name]
        if column.type != field.type:
            column = pc.cast(column, field.type, safe=False)
        arrays.append(column)

    return pa.Table.from_arrays(arrays, schema=target_schema)

## 5.4.4. Stream daily Parquet files into monthly outputs

In [ ]:
for _, month_row in months_df.iterrows():
    year = int(month_row["year"])
    month = int(month_row["month"])
    period = month_row["period"]
    output_name = f"{year}-{month}"
    output_path = os.path.join(
        g.Config.STAGE5_MONTHLY_FULL_PARQUET_PATH,
        output_name + ".parquet",
    )
    temporary_path = output_path + ".tmp"
    ua_json_path = os.path.join(
        g.Config.STAGE5_MONTHLY_FULL_JSON_UA_PATH,
        output_name + ".json",
    )

    month_tracker = completed_process_df[
        completed_process_df["input_date"].dt.to_period("M").eq(period)
    ]
    valid_paths = []
    schemas = []

    for _, process_row in month_tracker.iterrows():
        path = process_row["rejoin_path"]
        if pd.isna(path) or not str(path).strip() or not os.path.isfile(path):
            print(f"Skipping {process_row['input_path']}: daily Parquet is missing.")
            continue
        try:
            parquet_file = pq.ParquetFile(path)
            if parquet_file.metadata.num_rows == 0:
                print(f"Skipping {process_row['input_path']}: daily Parquet is empty.")
                continue
            schemas.append(selected_schema(path))
            valid_paths.append(path)
        except Exception as error:
            print(f"Skipping {process_row['input_path']}: invalid Parquet: {error}")

    if not valid_paths:
        print(f"Month {output_name}: no valid daily Parquet files.")
        continue

    if not os.path.isfile(output_path):
        target_schema = unify_selected_schemas(schemas)
        if os.path.isfile(temporary_path):
            os.remove(temporary_path)

        writer = None
        total_rows = 0
        successful = False
        try:
            writer = pq.ParquetWriter(
                temporary_path,
                target_schema,
                compression="zstd",
            )
            for path in valid_paths:
                try:
                    table = pq.read_table(path, columns=[
                        column for column in selected_columns
                        if column in pq.read_schema(path).names
                    ])
                    table = align_table_to_schema(table, target_schema)
                    writer.write_table(table)
                    total_rows += table.num_rows
                    print(f"[{output_name}] {os.path.basename(path)}: {table.num_rows} rows")
                except Exception as error:
                    print(f"[{output_name}] skipped {path}: {error}")

            successful = total_rows > 0
        finally:
            if writer is not None:
                writer.close()

        if successful:
            os.replace(temporary_path, output_path)
            print(f"Month {output_name}: wrote {total_rows} rows.")
        elif os.path.isfile(temporary_path):
            os.remove(temporary_path)
    else:
        print(f"Month {output_name}: monthly Parquet already exists.")

    if os.path.isfile(output_path) and not os.path.isfile(ua_json_path):
        try:
            ua_data = pd.read_parquet(
                output_path,
                filters=[("country", "==", "Ukraine")],
            )
            ua_data.to_json(
                ua_json_path,
                orient="records",
                force_ascii=False,
                date_format="iso",
            )
            print(f"Month {output_name}: Ukraine JSON contains {ua_data.shape[0]} rows.")
        except Exception as error:
            print(f"Month {output_name}: cannot create Ukraine JSON: {error}")

print("All done!")

---
© 2026 Yurii Kleban, Britta Rude. All rights reserved.